# ContrastPhys+ on PURE Dataset (Google Colab)
**Hardware Acceleration Guide:**
* **Recommended GPU:** **L4 GPU** (fastest & cost-effective) or **A100 GPU**.
* Set via **Runtime -> Change runtime type -> Hardware accelerator: L4 GPU**.
* Expected 30-epoch training time: **~15 minutes on L4**, **~8 minutes on A100** (vs ~3.75 hours on local Mac MPS).
* All checkpoints and results are automatically synced to Google Drive.

### 1. Mount Google Drive
Mount Drive so checkpoints, logs, and benchmark tables persist even if Colab disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/rPPG_Checkpoints", exist_ok=True)
os.makedirs("/content/drive/MyDrive/rPPG_Thesis_Results", exist_ok=True)
print("Google Drive mounted and backup folders ready!")

### 2. Clone Repository & Setup Working Directory

In [ ]:
%cd /content
!git clone https://github.com/drozanes/rPPG-Toolbox.git || (cd rPPG-Toolbox && git pull)
%cd /content/rPPG-Toolbox
!git checkout main
!git pull

### 3. Install Required Dependencies & Verify CUDA GPU

In [ ]:
!pip install -q ftfy yacs timm h5py wandb matplotlib scipy scikit-learn tabulate huggingface_hub

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No CUDA GPU detected! Go to Runtime -> Change runtime type -> Select GPU!")

### 3.1 Connect to Weights & Biases (WandB)
Logs loss curves, IPR progression, and test metrics live to the  dashboard.

In [ ]:
import wandb
# Enter your WandB API key when prompted (or pass it via wandb.login(key="..."))
wandb.login()
print("✅ Successfully authenticated with Weights & Biases!")

### 4. Fast Download of PURE Dataset via HuggingFace (~45 sec)
We download the dataset directly from HuggingFace to Colab NVMe SSD.

In [ ]:
import os
if not os.path.exists("./PURE_dataset") or len(os.listdir("./PURE_dataset")) < 5:
    print("Downloading PURE dataset from HuggingFace (WeiQian98/PURE)...")
    !huggingface-cli download WeiQian98/PURE --repo-type dataset --local-dir ./PURE_dataset --token 
    print("Download finished!")
else:
    print("PURE dataset already exists on local SSD!")

### 5. Fast Local Preprocessing for ContrastPhys (~3-4 min)
Preprocesses 128x128 face crops and DiffNormalized clips onto the fast local NVMe SSD.

In [ ]:
import glob
cached_files = glob.glob("PreprocessedData/PURE_608f362849/*.npy")
if len(cached_files) < 100:
    print("Running ContrastPhys preprocessing for PURE...")
    !python main.py --config_file configs/preprocess_configs/PURE_PREPROCESS_CONTRASTPHYS.yaml
    cached_files = glob.glob("PreprocessedData/PURE_608f362849/*.npy")

print(f"Preprocessed dataset ready! Total files: {len(cached_files)}")

### 6. Generate Paper-Calibrated Training Config (Contrast-Phys ECCV / TPAMI Setup)
Calibrated with `LR: 1e-5`, `S: 2` (4 quadrants), `MODEL_SELECTION: IPR`, and 30s evaluation window matching the ECCV 2022 / TPAMI 2024 benchmark.

In [ ]:
colab_yaml = """
BASE: ['']
TOOLBOX_MODE: "train_and_test"
TRAIN:
  BATCH_SIZE: 8  # 8 videos per batch gives 28 negative video pairs
  EPOCHS: 30
  PATIENCE: 15
  LR: 1e-5       # Paper specification (ECCV 2022): AdamW 1e-5 prevents latent collapse
  MODEL_FILE_NAME: PURE_ContrastPhys_Colab_Calibrated
  PLOT_LOSSES_AND_LR: True
  DATA:
    FS: 30
    DATASET: PURE
    DO_PREPROCESS: False
    DATA_FORMAT: NCDHW
    DATA_PATH: "./PURE_dataset"
    CACHED_PATH: "PreprocessedData"
    EXP_DATA_NAME: "PURE_954977d7f0"
    BEGIN: 0.0
    END: 0.8
    PREPROCESS:
      DATA_TYPE: ['Standardized']
      DATA_AUG: ['None']
      LABEL_TYPE: Raw
      DO_CHUNK: True
      CHUNK_LENGTH: 128
      CROP_FACE:
        DO_CROP_FACE: True
        BACKEND: 'HC'
        USE_LARGE_FACE_BOX: True
        LARGE_BOX_COEF: 1.5
        DETECTION:
          DO_DYNAMIC_DETECTION: False
          DYNAMIC_DETECTION_FREQUENCY: 30
          USE_MEDIAN_FACE_BOX: False
      RESIZE:
        H: 128
        W: 128
VALID:
  DATA:
    FS: 30
    DATASET: PURE
    DO_PREPROCESS: False
    DATA_FORMAT: NCDHW
    DATA_PATH: "./PURE_dataset"
    CACHED_PATH: "PreprocessedData"
    EXP_DATA_NAME: "PURE_954977d7f0"
    BEGIN: 0.8
    END: 1.0
    PREPROCESS:
      DATA_TYPE: ['Standardized']
      DATA_AUG: ['None']
      LABEL_TYPE: Raw
      DO_CHUNK: True
      CHUNK_LENGTH: 128
      CROP_FACE:
        DO_CROP_FACE: True
        BACKEND: 'HC'
        USE_LARGE_FACE_BOX: True
        LARGE_BOX_COEF: 1.5
        DETECTION:
          DO_DYNAMIC_DETECTION: False
          DYNAMIC_DETECTION_FREQUENCY: 30
          USE_MEDIAN_FACE_BOX: False
      RESIZE:
        H: 128
        W: 128
TEST:
  METRICS: ['MAE', 'RMSE', 'MAPE', 'Pearson', 'SNR', 'BA']
  USE_LAST_EPOCH: False
  DATA:
    FS: 30
    DATASET: PURE
    DO_PREPROCESS: False
    DATA_FORMAT: NCDHW
    DATA_PATH: "./PURE_dataset"
    CACHED_PATH: "PreprocessedData"
    EXP_DATA_NAME: "PURE_954977d7f0"
    BEGIN: 0.8
    END: 1.0
    PREPROCESS:
      DATA_TYPE: ['Standardized']
      LABEL_TYPE: Raw
      DO_CHUNK: True
      CHUNK_LENGTH: 128
      CROP_FACE:
        DO_CROP_FACE: True
        BACKEND: 'HC'
        USE_LARGE_FACE_BOX: True
        LARGE_BOX_COEF: 1.5
        DETECTION:
          DO_DYNAMIC_DETECTION: False
          DYNAMIC_DETECTION_FREQUENCY: 30
          USE_MEDIAN_FACE_BOX: False
      RESIZE:
        H: 128
        W: 128
DEVICE: cuda:0
NUM_OF_GPU_TRAIN: 1
LOG:
  PATH: runs/exp
MODEL:
  DROP_RATE: 0.2
  NAME: ContrastPhys
  PHYSNET:
    FRAME_NUM: 128
  CONTRAST_PHYS:
    S: 2                   # 2x2 spatial quadrants (paper specification)
    DELTA_T: 128           # Temporal window
    K: 4                   # 4 temporal clips per sub-region
    HIGH_PASS: 40.0        # physiological 40 bpm cutoff
    LOW_PASS: 250.0        # physiological 250 bpm cutoff
    ZERO_PAD: 2            # zero-padding factor for high-resolution PSD peak detection
    WEAK_SUPERVISION: True # Contrast-Phys+ weakly-supervised with GT signals
    MODEL_SELECTION: "IPR" # Model checkpoint selection by lowest Irrelevant Power Ratio (TPAMI 2024)
INFERENCE:
  BATCH_SIZE: 8
  EVALUATION_METHOD: "FFT"
  EVALUATION_WINDOW:
    USE_SMALLER_WINDOW: True  # 30-second sliding evaluation window (ECCV 2022 paper protocol)
    WINDOW_SIZE: 30
  MODEL_PATH: ""
"""

with open("configs/train_configs/PURE_CONTRASTPHYS_COLAB.yaml", "w") as f:
    f.write(colab_yaml)

print("Config created: configs/train_configs/PURE_CONTRASTPHYS_COLAB.yaml")

### 7. Execute Training & Testing with CUDA Multi-Worker Acceleration (~15 min on L4)
Executes ContrastPhys+ with automatic logging to WandB and local CSV results.

In [ ]:
!python main.py \n  --config_file configs/train_configs/PURE_CONTRASTPHYS_COLAB.yaml \n  --device cuda:0 \n  --num_workers 2 \n  --batch_size 8 \n  --backup_dir /content/drive/MyDrive/rPPG_Checkpoints

### 8. Backup All Models & Benchmark Results to Google Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/rPPG_Thesis_Results
!cp -r runs/exp/PURE_608f362849/PreTrainedModels /content/drive/MyDrive/rPPG_Thesis_Results/ || true
!cp rPPG_results_database.csv /content/drive/MyDrive/rPPG_Thesis_Results/ || true

import pandas as pd
if os.path.exists("rPPG_results_database.csv"):
    df = pd.read_csv("rPPG_results_database.csv")
    print("
=== Latest Results Database ===")
    display(df.tail(5))

print("
All checkpoints and results are securely saved to Google Drive!")

### 9. Download the Complete  Folder
Compresses all saved checkpoints, training logs, tensorboard files, and loss plots from  and downloads directly to your local computer (as well as copying to Google Drive).

In [ ]:
import os
import shutil
from google.colab import files

zip_output_path = "/content/runs_complete_archive.zip"
if os.path.exists("runs"):
    print("Zipping complete runs directory...")
    shutil.make_archive("/content/runs_complete_archive", "zip", "runs")
    print(f"Created archive: {zip_output_path} ({os.path.getsize(zip_output_path) / 1e6:.2f} MB)")
    
    # Copy to Google Drive
    if os.path.exists("/content/drive/MyDrive"):
        !cp /content/runs_complete_archive.zip /content/drive/MyDrive/rPPG_Thesis_Results/
        print("Copied archive to Google Drive: /content/drive/MyDrive/rPPG_Thesis_Results/runs_complete_archive.zip")
        
    # Download directly to your local computer browser
    print("Triggering browser download to your local machine...")
    files.download(zip_output_path)
else:
    print("runs directory not found!")